# 17.16 選最近的聲音碼不可微，編碼器怎樣學？


一個音框先被 encoder（編碼器）寫成 `[0.2,0.9]`，碼本只有 `[0,0]` 與 `[1,1]`。選最近者後，decoder（解碼器）收到 `[1,1]`；現在希望重建更接近原來兩個數字，該怎樣改 encoder？[12.18](https://birdhackor.github.io/tiny-perceptron-vlm/12.18.html)先固定碼本教查表，本節讓 encoder、碼本、decoder 都能更新。這裡量化的是聲音的中間特徵，和本章前面量化模型權重的對象不同。

最近碼以 `argmin` 選出距離最小的 ID。只要小幅移動後仍選同一碼，送進 decoder 的值完全不變；跨過選碼邊界又會突然跳動。硬選碼無法直接提供所需的 encoder 重建梯度。[17.14](https://birdhackor.github.io/tiny-perceptron-vlm/17.14.html)的 **STE（straight-through estimator，直通估計）**因此再次派上用場：前向真的用選到的碼，反向另外約定把 decoder 傳來的梯度接回 encoder。這是近似規則，不是 `argmin` 的真導數。

程式把 encoder 輸出叫 `z_e`、查表向量叫 `z_q`，以 `z_st = z_e + (z_q-z_e).detach()`接到 decoder。`detach()`保留相同數值，卻切斷這部分的反向關係。前向兩個 `z_e` 相消，得到 `z_q`；反向括號被視為常數，只剩外面的 `z_e`，梯度原樣通過。碼本也被括在切斷處，所以這份重建梯度不更新碼本。

碼本仍需要學，便另加一份碼本 loss，讓選中的向量靠近固定下來的 `z_e.detach()`；再加 **commitment loss（承諾代價）**，讓 encoder 靠近固定下來的 `z_q.detach()`，避免 encoder 的輸出只顧著漂移。下圖箭頭表示各份 loss 實際更新誰；承諾項的係數採0.25。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/omni-17.16-gradient-paths.svg")))

下面是完整的微型 **VQ-VAE（Vector Quantised Variational AutoEncoder，向量量化變分自編碼器）**訓練步驟：將連續特徵換成離散碼，再重建輸入。它依[VQ-VAE 原文 §3.1–3.2](https://arxiv.org/pdf/1711.00937v1)採三路 loss 更新，並用 MSE（mean squared error，平均平方誤差）作要最小化的重建代價。兩個 Linear 都人工設為單位矩陣，碼本也人工指定，尚非學到的聲音模型。每框獨立處理，因此先用 `valid` 選出有效框，padding 的大數值既不進 loss，也不進碼本使用計數。

輸出的「梯度量」是各參數梯度絕對值的總和，欄位依序為 encoder、碼本、decoder；沒有反向路徑的 `None` 也印成0。先分別檢查每份 loss，再對總和呼叫 `backward()`，最後 `step()`真正更新一次。


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

torch.set_num_threads(1)
torch.manual_seed(0)
encoder = nn.Linear(2, 2, bias=False)
decoder = nn.Linear(2, 2, bias=False)
codebook = nn.Parameter(torch.tensor([[0.0, 0.0], [1.0, 1.0]]))
with torch.no_grad():
    encoder.weight.copy_(torch.eye(2))
    decoder.weight.copy_(torch.eye(2))

x = torch.tensor([[[0.2, 0.9], [999.0, -999.0]]])
valid = torch.tensor([[True, False]])
target = x[valid]  # 先移除 padding，loss 與碼本計數都只看有效框。
z_e = encoder(target)
distance = (z_e[:, None, :] - codebook[None, :, :]).square().sum(-1)
ids = distance.argmin(-1)
z_q = codebook[ids]
z_st = z_e + (z_q - z_e).detach()

losses = {
    "重建": F.mse_loss(decoder(z_st), target),
    "碼本": F.mse_loss(z_q, z_e.detach()),
    "承諾": 0.25 * F.mse_loss(z_e, z_q.detach()),
}
params = [encoder.weight, codebook, decoder.weight]


def gradient_sizes(loss):
    grads = torch.autograd.grad(loss, params, retain_graph=True, allow_unused=True)
    return [0.0 if g is None else round(g.abs().sum().item(), 6) for g in grads]


def matrix(value):
    return [[round(v, 6) for v in row] for row in value.detach().tolist()]


print("有效框／碼／計數", len(target), ids.tolist(), torch.bincount(ids, minlength=2).tolist())
hard_loss = F.mse_loss(decoder(z_q), target)
print("只硬選的梯度量", gradient_sizes(hard_loss))
for name, loss in losses.items():
    print(name, round(loss.item(), 6), "梯度量", gradient_sizes(loss))

optimizer = torch.optim.SGD(params, lr=0.1)
optimizer.zero_grad()
sum(losses.values()).backward()
print("encoder總梯度", matrix(encoder.weight.grad))
optimizer.step()  # 真正做一次參數更新。
for name, value in zip(("encoder", "碼本", "decoder"), params):
    print("更新後" + name, matrix(value))

關鍵輸出為：

```text
有效框／碼／計數 1 [1] [0, 1]
只硬選的梯度量 [0.0, 0.9, 1.8]
重建 0.325 梯度量 [0.99, 0.0, 1.8]
碼本 0.325 梯度量 [0.0, 0.9, 0.0]
承諾 0.08125 梯度量 [0.2475, 0.0, 0.0]
encoder總梯度 [[0.12, 0.54], [0.015, 0.0675]]
更新後encoder [[0.988, -0.054], [-0.0015, 0.99325]]
更新後碼本 [[0.0, 0.0], [0.92, 0.99]]
更新後decoder [[0.92, -0.08], [-0.01, 0.99]]
```

`[0.2,0.9]`到兩碼的平方距離為0.85與0.65，所以選 ID 1。重建的平均平方誤差是 `(0.8²+0.1²)/2=0.325`；碼本項此刻同為0.325，承諾項則為 `0.25×0.325=0.08125`。它們數值相等或成比例，反向路徑卻不同。只硬查表時，decoder 和被選碼本可以有梯度，encoder 為0；加入 STE 後，重建梯度接到 encoder，碼本改由自己的 loss 更新。更新後只有碼本的第1列移到 `[0.92,0.99]`，未選中的第0列仍為零。

這一次 SGD 更新驗證誰在學，沒有證明音質或留出資料表現。正式訓練還要觀察有效框的碼本使用量：若長期幾乎全擠到少數碼，就是碼本塌縮的警訊；這個只有一框的 `[0,1]` 計數本身不能作此判斷。[EnCodec §3.2](https://arxiv.org/pdf/2210.13438v1)採 EMA（exponential moving average，指數移動平均）移動碼本，是另一種更新配方；本例沒有混用它。

codec 的 encoder、碼本、decoder 與前處理經開發集選定後，要先凍結，再產生 Talker 的訓練目標；否則同一 ID 的意義仍在移動。之後 Talker 的 optimizer 不含 codec，評估也不更新碼本或統計。codec 能重建輸入，與 Talker 能從條件猜出這些碼，是後續要分別驗收的工作。

練習只把承諾項係數 `0.25` 改成0，從頭執行。先預測碼本與 decoder 的這次更新相同，而 encoder 少掉承諾梯度；它的總梯度應變成 `[[0.16,0.72],[0.02,0.09]]`。再把 padding 兩個大數換成任意有限值，確認結果不變。這兩個改動分別檢查 loss 的分工與有效框的範圍。
